In [5]:
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()
database_url = os.getenv('DATABASE_URL')
engine = create_engine(database_url)
print("db 연결 성공!")

db 연결 성공!


# baseline_counts table

In [6]:
# 데이터 로드 

query = 'select * from ctgov.baseline_counts;'
df_baseline_counts = pd.read_sql(query, engine)

In [7]:
df_baseline_counts.head()

,id,nct_id,result_group_id,ctgov_group_code,units,scope,count
0,72197275,NCT00611403,644190387,BG000,Participants,overall,68
1,72197276,NCT00611403,644190388,BG001,Participants,overall,69
2,72197277,NCT00611403,644190389,BG002,Participants,overall,137
3,72197278,NCT00539864,644190390,BG000,Participants,overall,300
4,72197279,NCT00539864,644190391,BG001,Participants,overall,302


### 컬럼 설명
- result_group_id: 결과 보고시 구분된 그룹 id (연구내에서 환자들을 나눈 내부 그룹 번호)
- ctgov_group_code: clinicaltrails.gov 그룹 코드
- units : 수치의 단위 (participants는 사람 수)
- scope : 데이터 측정 범위(overall 특정시점 아닌 전체적인 집계)
- count : 실제 숫자값 (해당 그룹에 참여한 사람의 수)

In [8]:
df_baseline_counts.isnull().sum()

id                  0
nct_id              0
result_group_id     0
ctgov_group_code    0
units               0
scope               0
count               0
dtype: int64

In [9]:
df_baseline_counts.info()

<class 'pandas.DataFrame'>
RangeIndex: 229801 entries, 0 to 229800
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                229801 non-null  int64
 1   nct_id            229801 non-null  str  
 2   result_group_id   229801 non-null  int64
 3   ctgov_group_code  229801 non-null  str  
 4   units             229801 non-null  str  
 5   scope             229801 non-null  str  
 6   count             229801 non-null  int64
dtypes: int64(3), str(4)
memory usage: 12.3 MB


In [10]:
df_baseline_counts['units'].value_counts()

units
Participants           228451
Eyes                      178
eyes                       96
Teeth                      37
Clinics                    36
                        ...  
NCCLs restorations          1
worst eye                   1
Ears                        1
Joints                      1
ceramic restoration         1
Name: count, Length: 262, dtype: int64

In [11]:
# 불필요한 컬럼 삭제
df_baseline_counts = df_baseline_counts.drop(columns=['id', 'scope'])
# 중복 제거 및 식별자(nct_id)가 없는 행 삭제
df_baseline_counts = df_baseline_counts.dropna(subset=['nct_id']).drop_duplicates()

In [12]:
# 남은 컬럼 전처리 : units/ count/ result_group_id/ ctgov_guoup_code
# units: 데이터 필터링 - 참가자 단위만 
df_baseline_counts = df_baseline_counts[df_baseline_counts['units']=='Participants']
# count: nct_id별로 모든 참가자 수를 합산
df_baseline_counts = df_baseline_counts.groupby('nct_id')['count'].sum().reset_index()
# count: 컬럼명 알아보기 쉽게 변경
df_baseline_counts.rename(columns={'count':'overall_count'}, inplace=True)

In [13]:
# 전처리 후 테이블
df_baseline_counts.info()

<class 'pandas.DataFrame'>
RangeIndex: 76841 entries, 0 to 76840
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   nct_id         76841 non-null  str  
 1   overall_count  76841 non-null  int64
dtypes: int64(1), str(1)
memory usage: 1.2 MB


# baseline_measurements table

In [14]:
# 데이터 로드
query = """ 
select *
from ctgov.baseline_measurements;
"""
df_baseline_measurements = pd.read_sql(query, engine)


In [15]:
df_baseline_measurements.head()

,id,nct_id,result_group_id,ctgov_group_code,classification,category,title,description,units,param_type,...,dispersion_type,dispersion_value,dispersion_value_num,dispersion_lower_limit,dispersion_upper_limit,explanation_of_na,number_analyzed,number_analyzed_units,population_description,calculate_percentage
0,843780624,NCT03329027,643349750.0,BG000,Age,NaN,"Age, Customized",NaN,years,MEAN,...,FULL_RANGE,NaN,NaN,25.56,68.00,NaN,41.0,Participants,NaN,NaN
1,843780625,NCT03329027,643349751.0,BG001,Age,NaN,"Age, Customized",NaN,years,MEAN,...,FULL_RANGE,NaN,NaN,24.31,60.25,NaN,35.0,Participants,NaN,NaN
2,843780626,NCT03329027,643349752.0,BG002,Age,NaN,"Age, Customized",NaN,years,MEAN,...,FULL_RANGE,NaN,NaN,22.05,66.59,NaN,76.0,Participants,NaN,NaN
3,843780627,NCT03329027,643349753.0,BG003,Age,NaN,"Age, Customized",NaN,years,MEAN,...,FULL_RANGE,NaN,NaN,22.05,68.00,NaN,565.0,Participants,NaN,NaN
4,843780628,NCT03329027,643349750.0,BG000,NaN,Female,"Sex: Female, Male",NaN,Participants,COUNT_OF_PARTICIPANTS,...,NaN,NaN,NaN,NaN,NaN,NaN,41.0,Participants,NaN,NaN


In [16]:
df_baseline_measurements.info()

<class 'pandas.DataFrame'>
RangeIndex: 2802384 entries, 0 to 2802383
Data columns (total 22 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   id                      int64  
 1   nct_id                  str    
 2   result_group_id         float64
 3   ctgov_group_code        str    
 4   classification          str    
 5   category                str    
 6   title                   str    
 7   description             str    
 8   units                   str    
 9   param_type              str    
 10  param_value             str    
 11  param_value_num         float64
 12  dispersion_type         str    
 13  dispersion_value        str    
 14  dispersion_value_num    float64
 15  dispersion_lower_limit  float64
 16  dispersion_upper_limit  float64
 17  explanation_of_na       str    
 18  number_analyzed         float64
 19  number_analyzed_units   str    
 20  population_description  str    
 21  calculate_percentage    str    
dtypes: fl

In [17]:
# 필요한 컬럼 선택
select_col = [
    'nct_id', 'category', 'title', 'units', 'param_value_num', 
    'number_analyzed', 'number_analyzed_units'
]

df_baseline_measurements = df_baseline_measurements[select_col]
print(f'정리 후 컬럼수:{len(df_baseline_measurements.columns)}')
df_baseline_measurements.head()

정리 후 컬럼수:7


,nct_id,category,title,units,param_value_num,number_analyzed,number_analyzed_units
0,NCT03329027,NaN,"Age, Customized",years,44.39,41.0,Participants
1,NCT03329027,NaN,"Age, Customized",years,41.85,35.0,Participants
2,NCT03329027,NaN,"Age, Customized",years,41.31,76.0,Participants
3,NCT03329027,NaN,"Age, Customized",years,42.51,565.0,Participants
4,NCT03329027,Female,"Sex: Female, Male",Participants,18.00,41.0,Participants


### 컬럼 설명
- category: 세부 분류 (범주-male, female, white, asian..)
- title: 측정 항목의 제목 (age, gender, race, systolic blood pressure..)
- units: 수치의 단위 (param_value_num의 단위 - years, participants, mmHG..)
- param_value_num: 실제 측정된 수치값 (가장 중요한 컬럼)
- number_analyzed: 분석 대상 인원수 (해당 항목을 측정할 때 실제로 참여한 사람수)
- number_analyzed_units: 분석 인원의 단위 (보통 Participants..)

In [18]:
df_baseline_measurements.info()

<class 'pandas.DataFrame'>
RangeIndex: 2802384 entries, 0 to 2802383
Data columns (total 7 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   nct_id                 str    
 1   category               str    
 2   title                  str    
 3   units                  str    
 4   param_value_num        float64
 5   number_analyzed        float64
 6   number_analyzed_units  str    
dtypes: float64(2), str(5)
memory usage: 149.7 MB


In [19]:
df_baseline_measurements.isnull().sum()

nct_id                         0
category                 1003890
title                          0
units                        165
param_value_num             7780
number_analyzed             2584
number_analyzed_units          0
dtype: int64

### 1. number_analyzed_units

In [20]:
#  데이터 필터링 - participants만
df_baseline_measurements = df_baseline_measurements[df_baseline_measurements['number_analyzed_units'] == 'Participants'].dropna(subset=['param_value_num']).drop_duplicates()


In [21]:
df_baseline_measurements['number_analyzed_units'].value_counts()

number_analyzed_units
Participants    2653852
Name: count, dtype: int64

### 2. title

In [22]:
# 컬럼 값 확인 : 무려 5만8천개 항목들이 있음
df_baseline_measurements['title'].value_counts()

title
Race (NIH/OMB)                                                                           628753
Sex: Female, Male                                                                        427798
Ethnicity (NIH/OMB)                                                                      225312
Age, Categorical                                                                         207223
Region of Enrollment                                                                     183056
                                                                                          ...  
Blood Pressure Self-Monitoring (Self-Reported)                                                1
Count of Participants with Disease Present Prior to Stem Cell Transplant Conditioning         1
Participant Overall Satisfaction with Diet                                                    1
Diet Satisfaction- Cost Factor                                                                1
Diet Satisfaction-Planning & Prepa

In [23]:
# 필요한 내용만 찾기 (age, sex, gender)
# 'age'가 포함된 유니크한 값들만 추출(대소문자 구분안함)
age_titles = [t for t in df_baseline_measurements['title'].unique() if 'age' in str(t).lower()]
age_counts = df_baseline_measurements[df_baseline_measurements['title'].isin(age_titles)]['title'].value_counts()
print(f'Age 관련 항목 개수: {len(age_titles)}')
print(age_titles[:20])
print(age_counts.head(20))

Age 관련 항목 개수: 2844
['Age, Customized', 'Age, Continuous', 'Age, Categorical', 'Age Continuous', 'Type of Insurance Coverage', 'Average number of migraines attacks per month', 'Average number of migraine days per month', 'Stage of prostate cancer', 'Self-efficacy to Manage Chronic Disease Scale', 'Gestational age at randomization', 'International Federation of Gynecology and Obstetrics (FIGO) Stage Recurrent/Persistent', 'gestational age at birth', 'postmenstrual age at study entry', 'Age Categorical', 'Onset age', 'Melanoma Tumor Stage; Metastasis Classification at Study Entry', 'Weekly Mean of 24-Hour Average Pain', 'Brief Pain Inventory (BPI)-Severity Average Pain in the Last Week', 'BPI-Interference Average Score', 'End stage renal disease (ESRD) cause']
title
Age, Categorical            207223
Age, Continuous             174872
Age, Customized              62478
Age Continuous               12632
Age Categorical                577
Cancer Stage                   447
Stage           

In [24]:
# 'sex' 또는 'gender' 찾기
gender_titles = [t for t in df_baseline_measurements['title'].unique() if 'sex' in str(t).lower()]
gender_counts = df_baseline_measurements[df_baseline_measurements['title'].isin(gender_titles)]['title'].value_counts()
print(f'성별 관련 항목 개수: {len(gender_titles)}')
print(gender_titles[:20])
print(gender_counts.head(20))

성별 관련 항목 개수: 314
['Sex: Female, Male', 'Sex/Gender, Customized', 'Currently Sexually Active', 'Sexual Interest/Desire Inventory (SIDI)', 'Female Sexual Distress Scale-Revised (FSDS-R)', 'Pelvic Organ Prolapse/Incontinence Sexual Questionnaire IUGA-Revised (PISQ-IR)', 'Fetal sex female', 'Phase I: Youth sex', 'Phase II: Youth sex', 'Sexual Orientation', 'Sexually Active', 'Sexual orientation', 'Sexually active', 'Cluster level mean of any IPV (physical or sexual violence) in past 6 months', 'Cluster level mean of severe IPV (physical and/or sexual violence) in the past 6 months', 'Cluster level mean of sex work population size', 'Time-line Follow-back Alcohol and Sex (30 day)', 'Donor/Recipient Sex Match', 'Sexual-Orientation', 'Sex: Female, Male (PEP)']
title
Sex: Female, Male                                                                 427798
Sex/Gender, Customized                                                             18018
Sexual Orientation                                  

In [25]:
# 항목 이름 표준화 정렬 (age_categorical: 범주형 연령/ age_continusous: 연속형 연령)
df_baseline_measurements['title'] = df_baseline_measurements['title'].replace({
    'Age, Categorical': 'age_categorical',
    'Age Categorical': 'age_categorical',
    'Age, Continuous': 'age_continuous',
    'Age Continuous': 'age_continuous',
    'Sex: Female, Male': 'gender',
    'Gender': 'gender'
})

# 사용할 항목만 남기기
df_baseline_measurements = df_baseline_measurements[df_baseline_measurements['title'].isin(['age_categorical', 'age_continuous', 'gender'])]
df_baseline_measurements['title'].value_counts()

title
gender             435090
age_categorical    207800
age_continuous     187504
Name: count, dtype: int64

### 3. units

In [26]:
df_baseline_measurements['units'].value_counts()


units
Participants                      640621
years                             152255
Years                              29399
participants                        2223
months                              1286
                                   ...  
Age (years) at date of surgery         1
aggregate age                          1
YEAR                                   1
days of life                           1
mean years total population            1
Name: count, Length: 152, dtype: int64

In [27]:
df_baseline_measurements['units'].isnull().sum()

np.int64(61)

In [28]:
# 단위가 비어 있는 행은 삭제
df_baseline_measurements = df_baseline_measurements.dropna(subset=['units'])
df_baseline_measurements['units'].isnull().sum()

np.int64(0)

### 4. category

In [29]:
df_baseline_measurements['category'].value_counts()

category
Male                                 216879
Female                               216737
Between 18 and 65 years               69259
>=65 years                            68908
<=18 years                            68267
FEMALE                                   76
MALE                                     76
BTWN                                     52
GTE65                                    52
LTE18                                    51
Woman                                     6
Man                                       6
Famale                                    4
18-64 years                               4
65-84 years                               4
Between 18 and 65 Years                   4
≥18 to <50                                4
≥50 to <65                                4
≥65 to <75                                4
≥75                                       4
Non-binary                                3
< 75 years                                3
≥ 75 years             

In [30]:
df_baseline_measurements['category'].isnull().sum()

np.int64(189913)

In [31]:
# 항목 이름 표준화 정렬
df_baseline_measurements['category'] = df_baseline_measurements['category'].replace({
    'FEMALE': 'female', 'Female': 'female', 'Woman':'female', 'Famale':'female',
    'MALE': 'male', 'Male': 'male', 'Man':'male',
    '<=18 years': '<=18years', 
    'Between 18 and 65 years': '18-65years', '18-64 years': '18-65years', 'Between 18 and 65 Years': '18-65years',
    '>=65 years': '>=65years'
})
# 범주형 연령 처리 - category 없으면 삭제 (~: not의미. 여러행을 한꺼번에 처리할때 사용)
df_baseline_measurements = df_baseline_measurements[~(
    (df_baseline_measurements['title'] == 'age_categorical') & 
    (df_baseline_measurements['category'].isna()))]

# 연속형 연령 처리 - category 없으면 overall(전체평균 계산가능하므로)로 채움
df_baseline_measurements.loc[
    (df_baseline_measurements['title'] == 'age_continuous') & 
    (df_baseline_measurements['category'].isna()), 'category'
] = 'overall'

# 사용할 항목만 필터링
accepted = ['female', 'male', '<=18years', '18-65years', '>=65years', 'overall']
df_baseline_measurements = df_baseline_measurements[
    df_baseline_measurements['category'].isin(accepted)
]

df_baseline_measurements['category'].value_counts()

category
male          216961
female        216823
overall       187504
18-65years     69267
>=65years      68908
<=18years      68267
Name: count, dtype: int64

In [32]:
# 분석 목적에 따라 독립된 데이터프레임으로 데이터 분리
age_cat = df_baseline_measurements[df_baseline_measurements['title'] == 'age_categorical']
age_cont = df_baseline_measurements[df_baseline_measurements['title'] == 'age_continuous']
gender = df_baseline_measurements[df_baseline_measurements['title'] == 'gender']

# 피벗 테이블 생성 (category에 있던 female, male이 각각의 컬럼이 됨)
gender_pivot = gender.pivot_table(index='nct_id', columns='category', values='param_value_num', aggfunc='sum').reset_index()
age_cat_pivot = age_cat.pivot_table(index='nct_id', columns='category', values='param_value_num', aggfunc='sum').reset_index()

# 연속형 데이터 요약
# nct_id별 평균 나이 계산
mean_age = age_cont.groupby('nct_id')['param_value_num'].mean().reset_index().rename(columns={'param_value_num': 'mean_age'})
# 나이 분석에 참여한 총 인원수 확인
age_analyzed = age_cont.groupby('nct_id')['number_analyzed'].mean().reset_index().rename(columns={'number_analyzed': 'age_number_analyzed'})


In [33]:
# 4개 데이터프레임 합치기
from functools import reduce
baseline_features = reduce(lambda left, right : pd.merge(left, right, on='nct_id', how='outer'),
                           [age_cat_pivot, gender_pivot, mean_age, age_analyzed])

In [34]:
# 최종 데이터 셋 완성
df_baseline_features = pd.merge(baseline_features, df_baseline_counts, on='nct_id', how='left')
# 컬럼 재배치 및 컬럼 선택
df_baseline_features = df_baseline_features[['nct_id', 'overall_count', 'female', 'male', '<=18years', '18-65years', '>=65years', 'mean_age', 'age_number_analyzed']]


In [35]:
# 최종 데이터셋 저장
target_path = r'C:\dev\clinicaltrials-study\data\processed'
file_name = 'baseline_features_clean.csv'
full_save_path = os.path.join(target_path, file_name)
df_baseline_features.to_csv(full_save_path, index=False)

In [36]:
df_baseline_features.head()

,nct_id,overall_count,female,male,<=18years,18-65years,>=65years,mean_age,age_number_analyzed
0,NCT00000125,3272,1410.0,1862.0,NaN,NaN,NaN,55.4,50.666667
1,NCT00000134,558,46.0,512.0,0.0,558.0,0.0,NaN,NaN
2,NCT00000135,418,44.0,374.0,0.0,418.0,0.0,NaN,NaN
3,NCT00000136,468,40.0,428.0,0.0,468.0,0.0,NaN,NaN
4,NCT00000142,128,10.0,118.0,0.0,128.0,0.0,NaN,NaN


In [37]:
# 숫자형 데이터의 빈칸을 0으로 채우기
df_baseline_features = df_baseline_features.fillna(0)

In [38]:
df_baseline_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 75752 entries, 0 to 75751
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   nct_id               75752 non-null  str    
 1   overall_count        75752 non-null  int64  
 2   female               75752 non-null  float64
 3   male                 75752 non-null  float64
 4   <=18years            75752 non-null  float64
 5   18-65years           75752 non-null  float64
 6   >=65years            75752 non-null  float64
 7   mean_age             75752 non-null  float64
 8   age_number_analyzed  75752 non-null  float64
dtypes: float64(7), int64(1), str(1)
memory usage: 5.2 MB
